# LAB | Imbalanced

## Objetivo del lab

En este lab trabajamos con el dataset de **fraude en transacciones con tarjeta** del curso.

El objetivo es comprobar qué ocurre cuando la variable objetivo está desbalanceada y comparar varios enfoques:

1. Modelo base con los datos originales.
2. Modelo con **Oversampling**.
3. Modelo con **Undersampling**.
4. Modelo con **SMOTE**.

La variable objetivo es `fraud`:

- `0`: transacción legítima.
- `1`: transacción fraudulenta.

Como queremos predecir si una transacción es fraude o no, estamos ante un problema de **clasificación binaria**.


## 1. Importar librerías

Usamos `pandas` y `numpy` para manipulación de datos, `matplotlib` para visualizar, `scikit-learn` para el modelo y métricas, e `imblearn` para las técnicas de balanceo.


In [ ]:
# Librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Preparación de datos y modelo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Métricas de clasificación
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Técnicas para datos desbalanceados
try:
    from imblearn.over_sampling import RandomOverSampler, SMOTE
    from imblearn.under_sampling import RandomUnderSampler
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Necesitas instalar imbalanced-learn. Ejecuta: pip install imbalanced-learn"
    )

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)


## 2. Cargar los datos

El enunciado del lab indica trabajar con el dataset `card_transdata.csv`.

El código intenta primero cargar el archivo desde la carpeta local. Si no existe, lo carga desde la URL del curso.


In [ ]:
# Ruta local y URL del curso
local_path = "card_transdata.csv"
data_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv"

try:
    fraud = pd.read_csv(local_path)
    print("Dataset cargado desde archivo local.")
except FileNotFoundError:
    fraud = pd.read_csv(data_url)
    print("Dataset cargado desde la URL del curso.")

fraud.head()


## 3. Exploración inicial

Revisamos tamaño, tipos de datos, valores nulos y duplicados. Esta parte sigue el flujo habitual visto en clase antes de entrenar modelos.


In [ ]:
# Tamaño del dataset
fraud.shape


In [ ]:
print("Número de filas:", fraud.shape[0])
print("Número de columnas:", fraud.shape[1])


In [ ]:
# Información general del dataframe
fraud.info()


In [ ]:
# Valores nulos por columna
fraud.isna().sum()


In [ ]:
# Filas duplicadas
fraud.duplicated().sum()


In [ ]:
# Estadísticos descriptivos
fraud.describe().T


## 4. Distribución de la variable objetivo

La variable objetivo es `fraud`. Antes de entrenar el modelo revisamos cuántos casos hay de cada clase.


In [ ]:
# Distribución absoluta del target
fraud["fraud"].value_counts()


In [ ]:
# Distribución porcentual del target
fraud["fraud"].value_counts(normalize=True).mul(100).round(2)


In [ ]:
# Visualización de la distribución del target
fraud["fraud"].value_counts().sort_index().plot(kind="bar")
plt.title("Distribución de la variable objetivo")
plt.xlabel("Fraud: 0 = legítima, 1 = fraude")
plt.ylabel("Número de transacciones")
plt.xticks(rotation=0)
plt.show()


### Interpretación

Si la clase `0` aparece mucho más que la clase `1`, estamos ante un dataset **desbalanceado**.

En fraude financiero esto es normal: la mayoría de transacciones son legítimas y solo una minoría son fraudulentas.

Por eso, **accuracy no es suficiente** como métrica principal. Un modelo podría acertar muchas transacciones legítimas y aun así detectar mal el fraude.

En este lab nos fijamos especialmente en:

- `recall` de la clase fraude (`1`): cuántos fraudes reales detecta el modelo.
- `precision` de la clase fraude (`1`): de las transacciones marcadas como fraude, cuántas lo eran realmente.
- `f1-score`: equilibrio entre precision y recall.
- `balanced_accuracy`: accuracy corregida para clases desbalanceadas.
- `average_precision`: métrica útil cuando la clase positiva es minoritaria.


## 5. Separar variables predictoras y target

Las variables predictoras son todas las columnas excepto `fraud`.

La variable objetivo es `fraud`.


In [ ]:
X = fraud.drop("fraud", axis=1)
y = fraud["fraud"]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 6. Train-test split

Dividimos los datos en entrenamiento y test.

Usamos:

- 80% para entrenamiento.
- 20% para test.
- `stratify=y` para mantener la misma proporción de fraude/no fraude en ambos conjuntos.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


In [ ]:
print("Distribución en y_train:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nDistribución en y_test:")
print(y_test.value_counts(normalize=True).mul(100).round(2))


## 7. Escalado de variables

Usamos `StandardScaler` porque `LogisticRegression` puede beneficiarse de tener variables en escalas comparables.

Importante: el scaler se ajusta solo con `X_train` para evitar **data leakage**. Después se transforma `X_test` con ese mismo scaler.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 8. Función para entrenar y evaluar modelos

Creamos una función para repetir el mismo proceso con cada técnica de balanceo.

Así comparamos todos los modelos usando las mismas métricas.


In [ ]:
def evaluar_modelo(nombre_modelo, X_train_model, y_train_model, X_test_model, y_test_model):
    """
    Entrena una Logistic Regression y devuelve las principales métricas de clasificación.
    """
    modelo = LogisticRegression(max_iter=1000, random_state=42)
    modelo.fit(X_train_model, y_train_model)

    y_pred = modelo.predict(X_test_model)
    y_proba = modelo.predict_proba(X_test_model)[:, 1]

    resultados = {
        "modelo": nombre_modelo,
        "accuracy": accuracy_score(y_test_model, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test_model, y_pred),
        "precision_fraude": precision_score(y_test_model, y_pred, pos_label=1, zero_division=0),
        "recall_fraude": recall_score(y_test_model, y_pred, pos_label=1, zero_division=0),
        "f1_fraude": f1_score(y_test_model, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y_test_model, y_proba),
        "average_precision": average_precision_score(y_test_model, y_proba)
    }

    print("=" * 80)
    print(nombre_modelo)
    print("=" * 80)
    print("Matriz de confusión:")
    print(confusion_matrix(y_test_model, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test_model, y_pred, zero_division=0))

    return modelo, y_pred, resultados


## 9. Modelo base: Logistic Regression sin balancear

Primero entrenamos una `LogisticRegression` usando los datos originales.

Este será nuestro punto de comparación.


In [ ]:
modelo_base, pred_base, resultados_base = evaluar_modelo(
    "Logistic Regression - datos originales",
    X_train_scaled,
    y_train,
    X_test_scaled,
    y_test
)


### Interpretación del modelo base

Este modelo aprende con la distribución original del dataset.

Si el dataset está muy desbalanceado, puede tender a favorecer la clase mayoritaria (`0`: no fraude).

Por eso, aunque el accuracy pueda ser alto, debemos revisar especialmente el **recall de fraude**.


## 10. Oversampling

El **oversampling** consiste en aumentar la clase minoritaria duplicando ejemplos de esa clase hasta equilibrar el dataset de entrenamiento.

Lo aplicamos solo sobre el conjunto de entrenamiento, nunca sobre test.


In [ ]:
oversampler = RandomOverSampler(random_state=42)

X_train_over, y_train_over = oversampler.fit_resample(X_train_scaled, y_train)

print("Distribución original en train:")
print(y_train.value_counts())

print("\nDistribución tras oversampling:")
print(pd.Series(y_train_over).value_counts())


In [ ]:
modelo_over, pred_over, resultados_over = evaluar_modelo(
    "Logistic Regression - oversampling",
    X_train_over,
    y_train_over,
    X_test_scaled,
    y_test
)


### Interpretación de Oversampling

El oversampling suele mejorar el **recall de la clase minoritaria**, porque el modelo ve más ejemplos de fraude durante el entrenamiento.

Su posible desventaja es que puede aumentar el riesgo de overfitting, porque duplica observaciones existentes.


## 11. Undersampling

El **undersampling** consiste en reducir la clase mayoritaria hasta equilibrarla con la clase minoritaria.

También se aplica solo sobre el conjunto de entrenamiento.


In [ ]:
undersampler = RandomUnderSampler(random_state=42)

X_train_under, y_train_under = undersampler.fit_resample(X_train_scaled, y_train)

print("Distribución original en train:")
print(y_train.value_counts())

print("\nDistribución tras undersampling:")
print(pd.Series(y_train_under).value_counts())


In [ ]:
modelo_under, pred_under, resultados_under = evaluar_modelo(
    "Logistic Regression - undersampling",
    X_train_under,
    y_train_under,
    X_test_scaled,
    y_test
)


### Interpretación de Undersampling

El undersampling equilibra el dataset reduciendo la clase mayoritaria.

Su ventaja es que el entrenamiento puede ser más rápido y el modelo presta más atención a la clase minoritaria.

Su desventaja es que elimina muchos ejemplos de la clase mayoritaria, por lo que se puede perder información útil.


## 12. SMOTE

**SMOTE** significa **Synthetic Minority Oversampling Technique**.

A diferencia del oversampling simple, SMOTE no solo duplica ejemplos, sino que crea ejemplos sintéticos de la clase minoritaria a partir de vecinos cercanos.

Como SMOTE utiliza distancias, tiene sentido aplicarlo después del escalado.


In [ ]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Distribución original en train:")
print(y_train.value_counts())

print("\nDistribución tras SMOTE:")
print(pd.Series(y_train_smote).value_counts())


In [ ]:
modelo_smote, pred_smote, resultados_smote = evaluar_modelo(
    "Logistic Regression - SMOTE",
    X_train_smote,
    y_train_smote,
    X_test_scaled,
    y_test
)


### Interpretación de SMOTE

SMOTE suele ser más sofisticado que oversampling simple porque genera nuevas observaciones sintéticas en lugar de repetir las existentes.

Puede mejorar el recall de fraude, aunque también puede generar falsos positivos si la frontera entre clases no está bien separada.


## 13. Comparación final de resultados

Comparamos los cuatro enfoques:

1. Datos originales.
2. Oversampling.
3. Undersampling.
4. SMOTE.

La métrica más importante en este caso no es solo accuracy, sino especialmente el **recall de fraude** y el **F1-score de fraude**.


In [ ]:
resultados = pd.DataFrame([
    resultados_base,
    resultados_over,
    resultados_under,
    resultados_smote
])

resultados_ordenados = resultados.sort_values(
    by=["recall_fraude", "f1_fraude", "average_precision"],
    ascending=False
).reset_index(drop=True)

resultados_ordenados


In [ ]:
# Gráfico comparativo de métricas principales
metricas = ["balanced_accuracy", "precision_fraude", "recall_fraude", "f1_fraude", "average_precision"]

resultados_plot = resultados.set_index("modelo")[metricas]

ax = resultados_plot.plot(kind="bar", figsize=(12, 6))
plt.title("Comparación de modelos y técnicas de balanceo")
plt.ylabel("Score")
plt.xlabel("Modelo")
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 14. Matrices de confusión comparadas

Visualizamos las matrices de confusión para entender mejor los errores.

En fraude nos interesa especialmente reducir los **falsos negativos**, es decir, fraudes reales que el modelo clasifica como transacciones legítimas.


In [ ]:
predicciones = {
    "Original": pred_base,
    "Oversampling": pred_over,
    "Undersampling": pred_under,
    "SMOTE": pred_smote
}

for nombre, pred in predicciones.items():
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No fraude", "Fraude"])
    disp.plot(values_format="d")
    plt.title(f"Matriz de confusión - {nombre}")
    plt.show()


## 15. Conclusiones

### 1. ¿Está desbalanceado el target?

Sí. La clase `0` representa la mayoría de transacciones y la clase `1` representa una minoría. Por tanto, estamos trabajando con un problema de clasificación desbalanceada.

### 2. ¿Por qué no basta con accuracy?

Porque un modelo puede tener una accuracy alta prediciendo casi siempre la clase mayoritaria. En fraude, lo importante es detectar bien la clase minoritaria (`fraud = 1`).

### 3. ¿Qué métrica es más relevante?

Para este problema son especialmente importantes:

- `recall_fraude`: detectar el mayor número posible de fraudes reales.
- `f1_fraude`: equilibrio entre detectar fraudes y no generar demasiados falsos positivos.
- `average_precision`: útil con clases positivas minoritarias.

### 4. ¿Qué técnica funciona mejor?

La respuesta se obtiene al ejecutar la tabla comparativa. En general:

- El modelo original puede tener buena accuracy, pero peor sensibilidad hacia fraude.
- Oversampling suele mejorar el recall de fraude, aunque puede generar overfitting.
- Undersampling puede mejorar el recall, pero pierde información de la clase mayoritaria.
- SMOTE suele ser una alternativa más avanzada porque genera ejemplos sintéticos de fraude.

### 5. Respuesta final del lab

El mejor modelo debe elegirse priorizando el equilibrio entre `recall_fraude` y `f1_fraude`, no solo la accuracy.

En problemas de fraude, preferimos detectar la mayor cantidad posible de fraudes reales, aunque eso pueda aumentar algunos falsos positivos.


## 16. Respuesta breve para entregar

En este lab he comprobado que la variable objetivo `fraud` está desbalanceada, ya que la mayoría de transacciones son legítimas y solo una minoría son fraudulentas. Por este motivo, entrenar un modelo directamente sobre los datos originales puede provocar que el modelo favorezca la clase mayoritaria.

Primero entrené una `LogisticRegression` con los datos originales y evalué el resultado con métricas de clasificación. Después apliqué tres técnicas de balanceo sobre el conjunto de entrenamiento: `RandomOverSampler`, `RandomUnderSampler` y `SMOTE`. En todos los casos mantuve el conjunto de test sin modificar para evaluar correctamente el rendimiento real del modelo.

La comparación final se hizo usando `balanced_accuracy`, `precision`, `recall`, `f1-score`, `roc_auc` y `average_precision`. En este contexto, las métricas más importantes son el recall y el F1-score de la clase fraude, porque el objetivo principal es detectar correctamente las transacciones fraudulentas.
